In [ ]:
import math

from IPython.core.interactiveshell import InteractiveShell
InteractiveShell.ast_node_interactivity = "all"
import numpy as np
import sympy as sym

from rotation.symbolic import MASS, MIX, ANGLE
from rotation.solver import initialize, solve_mixing

In [ ]:
# Mass info from online PDG (in GeV)
# https://pdglive.lbl.gov/Particle.action?node=S009
# https://pdglive.lbl.gov/Particle.action?node=S014
PDG = {
    MASS["pi"]: 0.1349768,
    MASS["eta"]: 0.547862,
}

In [ ]:
def format_output(key, value):
    """Helps format solutions for printing"""
    if key in [MASS["pi"], MIX["pi"]]:
        value *= sym.Symbol(r" M_{\pi,\text{PDG}}") / PDG[MASS["pi"]]
    elif key in [MASS["eta"], MIX["eta"]]:
        value *= sym.Symbol(r" M_{\eta,\text{PDG}}") / PDG[MASS["eta"]]
    elif key in [MASS["phi"], MIX["phi"], MIX["pi-phi"], MIX["eta-phi"]]:
        value *= sym.Symbol(r" \text{ GeV}")
    elif key in ANGLE.values():
        value *= sym.Symbol(r" \pi") / math.pi
    return value

In [ ]:
argument_order, system_of_equations, partials, loaded_equation = initialize()

In [ ]:
# Fixed known parameters 
known = {
    MASS["pi"]: PDG[MASS["pi"]],
    MASS["eta"]: PDG[MASS["eta"]],
    MIX["phi"]: 1.0,
    MIX["pi-phi"]: 0.1,
    MIX["eta-phi"]: 0.3,
}

# These will be the unknowns we numerically solve for (the values are
# just initial guesses to get the solver going)
# Remember: the system of equations can only solve for up to 6 unknowns!
unknown = {
    MASS["phi"]: 1.0,
    MIX["pi"]: PDG[MASS["pi"]],
    MIX["eta"]: PDG[MASS["eta"]],
    ANGLE["alpha"]: 0.0,
    ANGLE["beta"]: 0.0,
    ANGLE["gamma"]: 0.0,
}

In [ ]:
try:
    # Solve system
    solved, raw = solve_mixing(
        system_of_equations,
        argument_order,
        unknown,
        known,
        partials,
        info=True,
    )

    # Printout the system with our known values, and unknowns blank
    print("Arguments input:")
    for k in argument_order:
        if k in known:
            sym.Eq(k, known[k], evaluate=False)
        else:
            sym.Eq(k, sym.Symbol("?"), evaluate=False)
    loaded_equation.subs(known)

    # Printout the system with our solved unknowns, and knowns blank
    print("Arguments solved:")
    for k, v in solved.items():
        v = format_output(k, v)
        sym.Eq(k, v, evaluate=False)

    print("The solved equation:")
    loaded_equation.subs(solved)

    print("Compare LHS and RHS evaluations:")
    sym.Eq(
        (1*loaded_equation.lhs.subs(known|solved)).evalf(),
        (1*loaded_equation.rhs.subs(known|solved)).evalf(),
        evaluate=False
    )

    print("Raw solver output:")
    print(raw)

except ValueError as e:
    print(e)
    print("The solver failed! Try other inputs.")